In [0]:
# Import all necessary libraries
import os
from dotenv import load_dotenv
load_dotenv()

In [0]:
# Connect the storage account to the Spark session

storage_account = os.getenv("storage_account")
application_id = os.getenv("application_id")
directory_id = os.getenv("directory_id")
secret_credentials = os.getenv("secret_credentials")

required_values = {
    "storage_account": storage_account,
    "application_id": application_id,
    "directory_id": directory_id,
    "secret_credentials": secret_credentials,
}

missing = [name for name, value in required_values.items() if not value]

if missing:
    raise ValueError(
        f"Missing environment variables: {', '.join(missing)}"
    )

account_host = f"{storage_account}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{account_host}",
    "OAuth",
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{account_host}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{account_host}",
    application_id,
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{account_host}",
    secret_credentials,
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{account_host}",
    f"https://login.microsoftonline.com/{directory_id}/oauth2/token",
)

print(f"Configured OAuth for {account_host}")

In [0]:
# Read the CSV file into a Spark DataFrame
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(
        "abfss://esoolistdata@esooliststorageaccount.dfs.core.windows.net/"
        "bronze/olist_customers_dataset.csv"
    )
)

display(df)